# Inter-Node GPU Transfer Benchmark: Object Store vs Compiled Graph vs NCCL/RDMA

Compares three transport methods for a **1 GB float32 tensor** between two A6000 GPUs on
separate nodes (`digilab-receiver` ↔ `digilab-transmit`, connected by a 400G Mellanox ConnectX-7
RoCE v2 fabric).

| Method | Transport | Expected peak |
|--------|-----------|---------------|
| Ray Object Store | TCP via eno2 (10 GbE management) | ~1 GB/s |
| Compiled Graph (default) | TCP via object store protocol | ~1–5 GB/s |
| Compiled Graph + NCCL/RDMA | RoCE v2 via mlx5_0 (400G) | ~40+ GB/s |

**Prerequisites** (run once per environment):
```bash
conda install ray[cgraph]   # or: pip install "ray[cgraph]"
```

**NCCL env** is injected by `cluster/ral_up.sh` at cluster start. Confirm with:
```bash
NCCL_DEBUG=INFO python -c "import torch; torch.distributed.init_process_group('nccl')"
# Look for: "mlx5_0:1<0>" in the NCCL output — that's your RDMA path.
```

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import pandas as pd
import ray
import seaborn as sns
import torch
from tqdm import tqdm

# Compiled graph warmup includes channel setup + NCCL group init on first execute,
# which can take 30–60 s for a 1 GB inter-node tensor. Must be set before
# experimental_compile() so the CompiledDAG picks up the value at construction time.
# Ray uses mixed-case here deliberately; noqa suppresses ruff's SIM112 uppercase check.
os.environ["RAY_CGRAPH_get_timeout"] = "120"  # noqa: SIM112

# Connect to the existing Ray cluster
ray.init(address="auto", ignore_reinit_error=True)

# 1 GB float32 tensor
TENSOR_SIZE_BYTES = 1 * 1024**3
TENSOR_SHAPE = (250_000_000,)

print(f"Ray Cluster Resources: {ray.available_resources()}")
print(f"Testing with Payload Size: {TENSOR_SIZE_BYTES / 1e9:.2f} GB")
print(f"Compiled graph get timeout: {os.environ['RAY_CGRAPH_get_timeout']} s")  # noqa: SIM112

# Set up plotting style for academic/HPC reporting
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

In [ ]:
@ray.remote(num_gpus=1)
class Sender:
    def __init__(self, shape):
        self.shape = shape
        self.tensor = torch.ones(self.shape, dtype=torch.float32, device="cuda")

    def get_data(self, trigger_input):  # <-- FIX: Added dummy parameter
        return self.tensor


@ray.remote(num_gpus=1)
class Receiver:
    def __init__(self):
        self.sink = None

    def consume_data(self, tensor):
        self.sink = tensor
        return True


# Instantiate the actors
sender = Sender.remote(TENSOR_SHAPE)
receiver = Receiver.remote()

# Warm up GPUs (pass a 0 as the dummy trigger)
ray.get(receiver.consume_data.remote(sender.get_data.remote(0)))
print("Actors initialized and warmed up.")

## §1 Baseline: Ray Object Store (TCP, inter-node)

Both actors carry `num_gpus=1` — with 1 GPU per node the scheduler places them on
separate nodes automatically. The object store path serialises the tensor through CPU
memory and sends it over the 10 GbE management interface (`eno2`).

In [ ]:
def benchmark_object_store(iterations=20):
    records = []
    print(f"Running Baseline (Object Store) for {iterations} iterations...")

    for i in range(iterations):
        start_time = time.perf_counter()

        # Standard Ray object store transfer — inter-node when actors land on separate nodes
        data_ref = sender.get_data.remote(0)
        ready_ref = receiver.consume_data.remote(data_ref)
        ray.get(ready_ref)

        end_time = time.perf_counter()
        iteration_time = end_time - start_time

        records.append(
            {
                "Method": "Ray Object Store (inter-node)",
                "Iteration": i + 1,
                "Latency (ms)": iteration_time * 1000,
                "Throughput (GB/s)": (TENSOR_SIZE_BYTES / 1e9) / iteration_time,
            }
        )

    return pd.DataFrame(records)


df_baseline = benchmark_object_store(iterations=20)

In [ ]:
# No NodeAffinitySchedulingStrategy — actors land on separate nodes automatically when the
# cluster has 1 GPU per node, measuring true inter-node 400G RDMA transfer performance.
# (The old same-node pin was measuring intra-node NVLink/PCIe, not the RDMA fabric.)
print(f"Sender:   {sender}")
print(f"Receiver: {receiver}")

## §2 Compiled Graph — default transport (no explicit NCCL)

Ray's compiled graph skips the Python-level serialisation overhead but still uses the
object store protocol under the hood unless a tensor transport is specified. This gives
the compiled-graph scheduling benefit without RDMA, so it's the right control to isolate
the NCCL/RDMA contribution in §3.

In [ ]:
from ray.dag.input_node import InputNode

# Compiled graph WITHOUT explicit NCCL transport — uses Ray's default protocol.
with InputNode() as inp:
    data = sender.get_data.bind(inp)
    result_default = receiver.consume_data.bind(data)

compiled_dag_default = result_default.experimental_compile()

# Warmup pass (not measured)
ray.get(compiled_dag_default.execute(0))


def benchmark_compiled_graph_default(iterations=20):
    records = []
    print(f"Running Compiled Graph (default transport) for {iterations} iterations...")
    for i in tqdm(range(iterations)):
        t0 = time.perf_counter()
        ray.get(compiled_dag_default.execute(i))
        elapsed = time.perf_counter() - t0
        records.append(
            {
                "Method": "Compiled Graph (default)",
                "Iteration": i + 1,
                "Latency (ms)": elapsed * 1000,
                "Throughput (GB/s)": (TENSOR_SIZE_BYTES / 1e9) / elapsed,
            }
        )
    return pd.DataFrame(records)


df_default = benchmark_compiled_graph_default(iterations=20)
print(f"Mean throughput: {df_default['Throughput (GB/s)'].mean():.2f} GB/s")

## §3 Compiled Graph + NCCL/RDMA (400G RoCE v2)

`.with_tensor_transport("nccl")` on the GPU→GPU edge routes the tensor directly through
NCCL's collective transport layer, which picks the RDMA path (`mlx5_0`) when the
`NCCL_IB_*` env vars are set. This bypasses CPU memory entirely for the data movement.

In [ ]:
from ray.dag.input_node import InputNode

# Compiled graph WITH NCCL tensor transport on the GPU-to-GPU edge.
with InputNode() as inp:
    data = sender.get_data.bind(inp)
    result = receiver.consume_data.bind(data.with_tensor_transport("nccl"))

compiled_dag_nccl = result.experimental_compile()

# Warmup pass (not measured)
ray.get(compiled_dag_nccl.execute(0))


def benchmark_compiled_graph_nccl(iterations=20):
    records = []
    print(f"Running Compiled Graph (NCCL/RDMA) for {iterations} iterations...")
    for i in tqdm(range(iterations)):
        t0 = time.perf_counter()
        ray.get(compiled_dag_nccl.execute(i))
        elapsed = time.perf_counter() - t0
        records.append(
            {
                "Method": "Compiled Graph (NCCL/RDMA)",
                "Iteration": i + 1,
                "Latency (ms)": elapsed * 1000,
                "Throughput (GB/s)": (TENSOR_SIZE_BYTES / 1e9) / elapsed,
            }
        )
    return pd.DataFrame(records)


df_nccl = benchmark_compiled_graph_nccl(iterations=20)
print(f"Mean throughput: {df_nccl['Throughput (GB/s)'].mean():.2f} GB/s")

df_results = pd.concat([df_baseline, df_default, df_nccl], ignore_index=True)
print("\nBenchmarking complete. Summary:")
print(
    df_results.groupby("Method")["Throughput (GB/s)"]
    .agg(["mean", "std"])
    .rename(columns={"mean": "mean (GB/s)", "std": "std (GB/s)"})
    .to_string()
)

In [ ]:
PALETTE = {
    "Ray Object Store (inter-node)": "#e74c3c",
    "Compiled Graph (default)": "#f39c12",
    "Compiled Graph (NCCL/RDMA)": "#2ecc71",
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f"Inter-Node GPU Transfer: {TENSOR_SIZE_BYTES / 1e9:.0f} GB float32  "
    f"(digilab-receiver ↔ digilab-transmit, 400G RoCE v2)",
    fontweight="bold",
)

# Throughput violin
sns.violinplot(
    data=df_results,
    x="Method",
    y="Throughput (GB/s)",
    ax=axes[0],
    palette=PALETTE,
    inner="quartile",
    order=list(PALETTE.keys()),
)
axes[0].set_title("Throughput Distribution (20 iterations)")
axes[0].set_ylabel("Throughput (GB/s)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=15)
axes[0].axhline(50.0, color="steelblue", linestyle=":", lw=1.0, label="400G theoretical (~50 GB/s)")
axes[0].axhline(1.25, color="gray", linestyle="--", lw=0.8, label="10 GbE limit (~1.25 GB/s)")
axes[0].legend(fontsize=8)

# Latency bar
sns.barplot(
    data=df_results,
    x="Method",
    y="Latency (ms)",
    ax=axes[1],
    palette=PALETTE,
    order=list(PALETTE.keys()),
    capsize=0.12,
    err_kws={"linewidth": 1.5},
)
axes[1].set_title("Mean Latency per Transfer (lower is better)")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

print("\n--- Summary Statistics ---")
summary = (
    df_results.groupby("Method")[["Throughput (GB/s)", "Latency (ms)"]]
    .agg(["mean", "std", "min", "max"])
    .reindex(list(PALETTE.keys()))
)
print(summary.to_string())

# Speedup factors vs object store baseline
base_mean = df_results[df_results["Method"] == "Ray Object Store (inter-node)"][
    "Throughput (GB/s)"
].mean()
print("\n--- Speedup vs Object Store Baseline ---")
for method in list(PALETTE.keys())[1:]:
    m = df_results[df_results["Method"] == method]["Throughput (GB/s)"].mean()
    print(f"  {method}: {m / base_mean:.1f}×")

## §4 Interpreting results & NCCL_DEBUG

### What to expect
| Transport | Mechanism | Bottleneck |
|-----------|-----------|------------|
| Object store | TCP → CPU RAM → TCP | 10 GbE NIC (`eno2`), ~1 GB/s |
| Compiled graph (default) | Same serialisation, less scheduling overhead | 10 GbE, ~1–5 GB/s |
| NCCL/RDMA | `mlx5_0` RDMA write (zero-copy GPU→GPU) | PCIe or 400G fabric, target >40 GB/s |

If NCCL/RDMA throughput is **not** significantly higher than the object store baseline,
check the NCCL transport selection — it may have fallen back to TCP.

### Compiled graph timeout
`RAY_CGRAPH_get_timeout` (set to 120 s in cell 1) controls how long the calling process
waits for each `compiled_dag.execute()` to return a result. The default is 10 s, which
is too short for:
- **First execution of default transport**: channel setup + 1 GB inter-node transfer (~5–15 s)
- **First execution of NCCL transport**: NCCL process group init + RDMA handshake (~15–60 s)

Subsequent iterations are fast once channels are warm. If you still see timeouts, raise
the value further: `os.environ["RAY_CGRAPH_get_timeout"] = "300"` and re-run cells 1–9.

### Reading NCCL_DEBUG=INFO output
After `cluster/ral_up.sh` restarts the cluster, the DDP init log will contain lines like:
```
NCCL INFO Channel 00/02: 0[0] -> 1[0] via P2P/IB/SL 0/5/5
NCCL INFO Using network IB
```
`P2P/IB` confirms RDMA (InfiniBand / RoCE). If you see `P2P/SHM` it's shared memory
(intra-node — wrong). If you see `P2P/NET/Socket` it's TCP fallback (check NCCL vars).

### Tuning knobs
```bash
# Override per-run without editing ral_up.sh:
NCCL_IB_HCA=mlx5_0 NCCL_IB_GID_INDEX=3 bash cluster/ral_up.sh --no-sync

# Increase NCCL buffer for large tensors (optional):
export NCCL_BUFFSIZE=8388608   # 8 MB (default 4 MB)
```